# 00. PyTorch Fundamentals Exercises

### 1. Documentation reading 

A big part of deep learning (and learning to code in general) is getting familiar with the documentation of a certain framework you're using. We'll be using the PyTorch documentation a lot throughout the rest of this course. So I'd recommend spending 10-minutes reading the following (it's okay if you don't get some things for now, the focus is not yet full understanding, it's awareness):
  * The documentation on [`torch.Tensor`](https://pytorch.org/docs/stable/tensors.html#torch-tensor).
  * The documentation on [`torch.cuda`](https://pytorch.org/docs/master/notes/cuda.html#cuda-semantics).



In [1]:
# No code solution (reading)

**Ghi chú sau khi đọc tài liệu:**

* **`torch.Tensor`**: là mảng nhiều chiều chứa các phần tử cùng một kiểu dữ liệu (`dtype`, ví dụ `torch.float32`, `torch.int64`).
  Mỗi tensor có `shape`, `dtype` và `device` (CPU hay GPU). Các phương thức kết thúc bằng `_` (ví dụ `add_()`) thay đổi tensor tại chỗ (in-place).
* **CUDA semantics**: `torch.cuda` theo dõi GPU đang được chọn; tensor tạo ra trên thiết bị nào thì phép tính trên nó chạy ở thiết bị đó.
  Không thể tính toán giữa tensor ở CPU và tensor ở GPU (hoặc hai GPU khác nhau) mà phải chuyển về cùng thiết bị bằng `.to(device)`.
  Các phép tính trên GPU chạy bất đồng bộ (asynchronous), vì vậy khi đo thời gian cần gọi `torch.cuda.synchronize()`.

### 2. Create a random tensor with shape `(7, 7)`.


In [2]:
# Import torch
import torch

# Create random tensor
X = torch.rand(size=(7, 7))
X, X.shape

(tensor([[0.0618, 0.4931, 0.0546, 0.0949, 0.4396, 0.4077, 0.5357],
         [0.7639, 0.0463, 0.9524, 0.4139, 0.0841, 0.7851, 0.7603],
         [0.4116, 0.6786, 0.6859, 0.8765, 0.9120, 0.1169, 0.2330],
         [0.6237, 0.1210, 0.3992, 0.6637, 0.5776, 0.4386, 0.5226],
         [0.0155, 0.1843, 0.8881, 0.1327, 0.8069, 0.7135, 0.5640],
         [0.3725, 0.1392, 0.2753, 0.2614, 0.0960, 0.1337, 0.3557],
         [0.8685, 0.9891, 0.8204, 0.3012, 0.1911, 0.4211, 0.7387]]),
 torch.Size([7, 7]))

### 3. Perform a matrix multiplication on the tensor from 2 with another random tensor with shape `(1, 7)` (hint: you may have to transpose the second tensor).

In [3]:
# Create another random tensor
Y = torch.rand(size=(1, 7))
# Z = torch.matmul(X, Y) # will error because of shape issues: (7, 7) @ (1, 7)

# Perform matrix multiplication (transpose Y so shapes line up: (7, 7) @ (7, 1) -> (7, 1))
Z = torch.matmul(X, Y.T)
Z, Z.shape

(tensor([[1.0561],
         [2.2450],
         [2.3196],
         [2.1799],
         [2.0826],
         [0.8773],
         [1.9902]]),
 torch.Size([7, 1]))

### 4. Set the random seed to `0` and do 2 & 3 over again.

The output should be:
```
(tensor([[1.8542],
         [1.9611],
         [2.2884],
         [3.0481],
         [1.7067],
         [2.5290],
         [1.7989]]), torch.Size([7, 1]))
```

In [4]:
# Set manual seed
torch.manual_seed(0)

# Create two random tensors
X = torch.rand(size=(7, 7))
Y = torch.rand(size=(1, 7))

# Matrix multiply tensors
Z = torch.matmul(X, Y.T)
Z, Z.shape

(tensor([[1.8542],
         [1.9611],
         [2.2884],
         [3.0481],
         [1.7067],
         [2.5290],
         [1.7989]]),
 torch.Size([7, 1]))

### 5. Speaking of random seeds, we saw how to set it with `torch.manual_seed()` but is there a GPU equivalent? (hint: you'll need to look into the documentation for `torch.cuda` for this one)
  * If there is, set the GPU random seed to `1234`.

In [5]:
# Set random seed on the GPU
# torch.cuda.manual_seed() is the GPU equivalent of torch.manual_seed()
# (it is safe to call even without a GPU, the call is silently ignored)
torch.cuda.manual_seed(1234)


### 6. Create two random tensors of shape `(2, 3)` and send them both to the GPU (you'll need access to a GPU for this). Set `torch.manual_seed(1234)` when creating the tensors (this doesn't have to be the GPU random seed). The output should be something like:

```
Device: cuda
(tensor([[0.0290, 0.4019, 0.2598],
         [0.3666, 0.0583, 0.7006]], device='cuda:0'),
 tensor([[0.0518, 0.4681, 0.6738],
         [0.3315, 0.7837, 0.5631]], device='cuda:0'))
```

In [6]:
# Set random seed
torch.manual_seed(1234)

# Check for access to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Create two random tensors on GPU
# Note: this machine has no NVIDIA GPU so the tensors will stay on the CPU,
# the same code sends them to the GPU when one is available.
tensor_A = torch.rand(size=(2, 3)).to(device)
tensor_B = torch.rand(size=(2, 3)).to(device)
tensor_A, tensor_B

Device: cpu


(tensor([[0.0290, 0.4019, 0.2598],
         [0.3666, 0.0583, 0.7006]]),
 tensor([[0.0518, 0.4681, 0.6738],
         [0.3315, 0.7837, 0.5631]]))


### 7. Perform a matrix multiplication on the tensors you created in 6 (again, you may have to adjust the shapes of one of the tensors).

The output should look like:
```
(tensor([[0.3647, 0.4709],
         [0.5184, 0.5617]], device='cuda:0'), torch.Size([2, 2]))
```

In [7]:
# Perform matmul on tensor_A and tensor_B
# tensor_C = torch.matmul(tensor_A, tensor_B) # shapes (2, 3) @ (2, 3) don't line up
tensor_C = torch.matmul(tensor_A, tensor_B.T) # (2, 3) @ (3, 2) -> (2, 2)
tensor_C, tensor_C.shape

(tensor([[0.3647, 0.4709],
         [0.5184, 0.5617]]),
 torch.Size([2, 2]))

### 8. Find the maximum and minimum values of the output of 7.

In [8]:
# Find max
max_value = torch.max(tensor_C)

# Find min
min_value = torch.min(tensor_C)
max_value, min_value

(tensor(0.5617), tensor(0.3647))

### 9. Find the maximum and minimum index values of the output of 7.

In [9]:
# Find arg max
arg_max = torch.argmax(tensor_C)


# Find arg min
arg_min = torch.argmin(tensor_C)
arg_max, arg_min

(tensor(3), tensor(0))


### 10. Make a random tensor with shape `(1, 1, 1, 10)` and then create a new tensor with all the `1` dimensions removed to be left with a tensor of shape `(10)`. Set the seed to `7` when you create it and print out the first tensor and it's shape as well as the second tensor and it's shape.

The output should look like:

```
tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]]) torch.Size([1, 1, 1, 10])
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513]) torch.Size([10])
```

In [10]:
# Set seed
torch.manual_seed(7)

# Create random tensor
tensor_D = torch.rand(size=(1, 1, 1, 10))

# Remove single dimensions
tensor_E = tensor_D.squeeze()

# Print out tensors and their shapes
print(tensor_D, tensor_D.shape)
print(tensor_E, tensor_E.shape)

tensor([[[[0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297,
           0.3653, 0.8513]]]]) torch.Size([1, 1, 1, 10])
tensor([0.5349, 0.1988, 0.6592, 0.6569, 0.2328, 0.4251, 0.2071, 0.6297, 0.3653,
        0.8513]) torch.Size([10])
